# PKG Attrition v13 · RM report builder

Builds a self-contained, interactive HTML report for RM managers from the saved results of the model notebook
(`m4_results.pkl` and the `figdata/` folder). Nothing is recomputed and no database is read: the report can be
rebuilt at any time in about a minute. Charts are Plotly and the Plotly library is embedded in the file, so the
report opens offline and can be emailed as a single file.

Set `RUN_DIR` to the model run's output folder and run both cells.

In [ ]:
# =====================================================================
# 0 · CONFIG
# =====================================================================
RUN_DIR   = "../metrics/attrition/v13/m4"       # output folder of the model notebook
PREFIX    = "m4"                                # file prefix used by that run
OUT_FILE  = "PKG_Attrition_RM_report.html"      # written inside RUN_DIR
TITLE     = "Deposit attrition early warning"
SUBTITLE  = "Which Treasury Management clients are about to move their money, and how early we can see it"
K_MAIN, A_MAIN = 1000, 0.5                      # operating point used in the model run
CAP_CREDIT = 250e6
SAVE_RATES = [0.01, 0.05, 0.10, 0.20]
EMBED_PLOTLY = True                             # True = works offline (~4 MB file); False = loads Plotly from the internet
EXAMPLES_SKIP = []                              # example numbers to leave out, e.g. [1]

In [ ]:
# =====================================================================
# 1 · BUILD THE REPORT
# =====================================================================
import numpy as np, pandas as pd, pickle, pathlib, re, html, datetime
for _a, _t in {"object0": np.object_, "bool8": np.bool_}.items():
    if not hasattr(np, _a): setattr(np, _a, _t)
try:
    import plotly, plotly.graph_objects as go, plotly.io as pio
    from plotly.subplots import make_subplots
    from plotly.offline import get_plotlyjs
except ImportError:
    raise ImportError("Plotly is not installed in this kernel: run  %pip install plotly  and restart the kernel")

RUN = pathlib.Path(RUN_DIR); FIGD = RUN / "figdata"
with open(RUN / f"{PREFIX}_results.pkl", "rb") as fh: BUNDLE = pickle.load(fh)
R = BUNDLE["R"]
def rd(name):
    p = FIGD / f"{PREFIX}_{name}.csv"
    return pd.read_csv(p) if p.exists() else None
LONG = rd("10_examples_long"); BV = rd("9d_leavers_vs_stayers_by_month"); LIFTM = rd("9a_lift_by_month")

# ── palette, formatting ──────────────────────────────────────────────
C = dict(navy="#0b2545", blue="#1d4e89", sky="#4f9dde", teal="#12a39a", orange="#f28c28", red="#d1495b",
         grey="#8d99ae", light="#f3f6fa", ink="#1f2933", gold="#e0b43a")
SEQ = [C["blue"], C["orange"], C["teal"], C["red"], C["grey"], C["gold"], C["sky"]]
def usd(v, d=1):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "—"
    s = "−" if v < 0 else ""; v = abs(v)
    return f"{s}${v / 1e9:,.{d}f}bn" if v >= 1e9 else f"{s}${v / 1e6:,.{d}f}m" if v >= 1e6 else f"{s}${v / 1e3:,.0f}k" if v >= 1e4 else f"{s}${v:,.0f}"
pct = lambda v, d=0: "—" if v is None or pd.isna(v) else f"{v:.{d}%}"
esc = html.escape

# ── plain-language names ─────────────────────────────────────────────
BASE = {
    "active_days_in": "days with money coming in", "active_days_out": "days with money going out",
    "ntxn_all_in": "incoming payments", "ntxn_all_out": "outgoing payments",
    "ntxn_ext_in": "incoming payments from clients of other banks", "ntxn_ext_out": "payments to clients of other banks",
    "ntxn_int_in": "incoming payments from other PNC clients", "ntxn_int_out": "payments to other PNC clients",
    "amt_all_in": "dollars coming in", "amt_all_out": "dollars going out", "amt_ext_in": "dollars received from other banks' clients",
    "amt_ext_out": "dollars paid to other banks' clients", "amt_int_in": "dollars received from other PNC clients",
    "amt_int_out": "dollars paid to other PNC clients", "ncpty_all_in": "number of payers", "ncpty_all_out": "number of payees",
    "ncpty_ext_in": "payers at other banks", "ncpty_ext_out": "payees at other banks", "ncpty_int_in": "payers who bank at PNC",
    "ncpty_int_out": "payees who bank at PNC", "n_fi_out": "banks the client pays into", "n_fi_in": "banks the client receives from",
    "n_fi_ach_out": "banks the client pays into by ACH", "n_fi_ach_in": "banks the client receives from by ACH",
    "net_flow_all": "net flow (money in minus money out)", "net_flow_ext": "net flow with other banks' clients",
    "passthrough_ratio": "share of incoming money paid straight back out", "selfpay_share_out": "share of outgoing money sent to its own accounts elsewhere",
    "selfpay_amt_out": "money sent to its own accounts at other banks", "avg_ticket_in": "average size of incoming payments",
    "avg_ticket_out": "average size of outgoing payments", "share_WIRE_in": "share of money received by wire",
    "share_CHECK_in": "share of money received by check", "share_ACH_in": "share of money received by ACH",
    "share_WIRE_out": "share of money paid by wire", "share_CHECK_out": "share of money paid by check", "share_ACH_out": "share of money paid by ACH",
    "rec_broken_in": "regular payers who skipped this month", "rec_broken_out": "regular payees skipped this month",
    "rec_broken_share_out": "share of regular payments that stopped", "rec_standing_out": "regular payees",
    "top1_share_in": "share of receipts from the single largest payer", "hhi_in": "concentration of receipts",
    "new_cpty_ext_in": "new payers at other banks", "lost_cpty_ext_in": "payers lost", "reactivated_cpty_ext_in": "returning payers",
    "fi_rec_broken_out": "regular banking relationships that stopped", "pay_log_flow": "overall payment activity",
    "dep_log_bal": "balance", "dep_log_norm": "normal balance", "dep_bal_vs_norm": "balance compared with normal",
    "dep_d1": "balance change vs last month", "dep_d3": "balance change vs the last 3 months", "dep_d6": "balance change vs 4–6 months earlier",
    "dep_d12": "balance change vs a year earlier", "dep_drawdown": "balance compared with its 12-month peak",
    "dep_vol6": "balance volatility", "dep_range": "swing in daily balance within the month",
    "dep_eom_vs_avg": "month-end balance below the monthly average (money leaving during the month)",
    "dep_siblings": "number of other entities in the same relationship",
    "dep_sib_d6": "balances of other entities in the same relationship", "dep_live_d6": "number of live accounts (change)",
    "dep_n_accts": "number of accounts", "dep_n_live": "number of live accounts", "dep_tenure": "time as a client",
    "dep_at_start": "long-standing client", "dep_earnings_credit_rate": "earnings credit rate", "dep_curr_int_rate": "interest rate paid",
    "dep_closed_3m": "accounts closed in the last 3 months", "dep_opened_3m": "accounts opened in the last 3 months",
}
SUFFIX = {"d6": "change vs 4–6 months earlier", "d3": "change vs the last 3 months", "peer": "compared with similar-sized clients"}
def plain(f, fallback=None):
    base, _, tr = str(f).partition("__")
    b = BASE.get(base)
    if b is None:
        b = re.sub(r"\s*\(.*?\)", "", str(fallback or base.replace("_", " "))).split(" — ")[0].strip()
    return b + (f" ({SUFFIX[tr]})" if tr in SUFFIX else "")
SETS = {"payment · flow": "Payment volumes", "payment · churn": "Changes in who they pay and receive from",
        "payment · bank churn": "Changes in the banks they deal with", "payment · rails": "Payment methods (wire, ACH, check)",
        "payment · recurring": "Regular payments", "payment · banks": "Banks involved in their payments",
        "payment · sector": "Industries of their counterparties", "payment · composition": "Mix of counterparties",
        "payment · self-pay": "Transfers to their own accounts elsewhere", "payment · breadth": "Number of counterparties",
        "payment · concentration": "Concentration of payments", "payment · activity status": "Whether payments stopped",
        "deposit · balance volatility": "Balance swings", "deposit · relationship": "Other entities in the relationship",
        "deposit · pricing": "Pricing (earnings credit, rates)", "deposit · balance change": "Balance trend",
        "deposit · accounts": "Accounts opened and closed", "deposit · product mix": "Deposit product mix",
        "deposit · tenure": "Time as a client", "deposit · balance level": "Balance compared with normal",
        "size anchor": "Client size", "missing share": "Missing-data patterns"}
def to_display(var, v):
    """Model units → units an RM reads: % change for log ratios, points for shares."""
    base = str(var).split("__")[0]
    if v is None or pd.isna(v): return np.nan
    if base.startswith("dep_live"): return v
    if re.search(r"(share|hhi|retention|entropy|coverage|ratio)", base) and base != "dep_bal_vs_norm": return v * 100
    if base.startswith("net_flow"): return v
    return (np.exp(v) - 1) * 100
def unit(var):
    base = str(var).split("__")[0]
    if base.startswith("dep_live"): return "accounts"
    if re.search(r"(share|hhi|retention|entropy|coverage|ratio)", base) and base != "dep_bal_vs_norm": return "percentage points"
    if base.startswith("net_flow"): return "index (log scale)"
    return "% change"
def play(var):
    v = str(var)
    if v.startswith("dep_sib"):
        return "Relationship risk: other entities in this relationship are already moving money. Talk to the relationship's decision maker, not only this entity."
    if re.search(r"(n_fi|^fi_|selfpay|top_fi|share_WIRE|share_ACH|share_CHECK|n_rails)", v):
        return "Competitive threat: money appears to be moving to another bank. Compare pricing, earnings credit and services with the likely competitor; prepare a retention offer."
    if re.search(r"(_in$|_in__|active_days_in|ncpty_.*_in)", v):
        return "Receipts are drying up: the client's own customers are paying it less, or paying into another bank. Ask about collections, lockbox and receivables arrangements."
    if v.startswith("dep_"):
        return "Balance drawdown: deposits are falling ahead of the payments. Ask about cash plans; offer sweep, money-market or liquidity options to keep the operating balance here."
    return "Business activity is shrinking: fewer payments and fewer trading partners. Check the client's business; offer credit, working-capital or payment-product support."

# ── charts ───────────────────────────────────────────────────────────
def show(fig, h=420):
    fig.update_layout(template="plotly_white", height=h, margin=dict(l=60, r=30, t=60, b=60),
                      font=dict(family="Segoe UI, Helvetica, Arial", size=13, color=C["ink"]),
                      title_font=dict(size=16, color=C["navy"]), legend=dict(orientation="h", y=-0.18, x=0),
                      hoverlabel=dict(font_size=12), colorway=SEQ)
    return "<div class='chart'>" + pio.to_html(fig, include_plotlyjs=False, full_html=False,
                                                config={"displaylogo": False, "responsive": True}) + "</div>"
def table(df, cols, heads, fmt):
    rows = "".join("<tr>" + "".join(f"<td>{fmt[c](r[c]) if c in fmt else esc(str(r[c]))}</td>" for c in cols) + "</tr>"
                   for _, r in df.iterrows())
    return f"<table class='t'><thead><tr>{''.join(f'<th>{h}</th>' for h in heads)}</tr></thead><tbody>{rows}</tbody></table>"

MAIN = "tree" if "tree" in set(R["acc"].model) else "simple"
acc, val = R["acc"], R["val"]
_pm = acc[(acc.model == MAIN) & (acc.alpha == A_MAIN) & (acc.calls == K_MAIN)].iloc[0]
_ps = acc[(acc.model == "simple") & (acc.alpha == A_MAIN) & (acc.calls == K_MAIN)].iloc[0]
HV = R["headline"].reset_index(drop=True)
hv = lambda key: float(HV.loc[HV.dollars.str.contains(key, regex=False), "reached_per_year"].iloc[0]) if HV.dollars.str.contains(key, regex=False).any() else np.nan
top5, capd, excl = hv("largest 5"), hv("capped"), hv("not losses")
lo = np.nanmin([top5, capd])
AR = R["at_risk_tab"].copy(); f_cli = R["exits_per_year"] / AR.clients.sum(); AR["clients_year"] = AR.clients * f_cli
big_share = AR.loc[AR.band.astype(str) == "$10m+", "per_year"].sum() / AR.per_year.sum()
lead_all = R["lead"]; lead1 = float(lead_all.loc[lead_all.lead == 1, "share_clients"].sum()) if len(lead_all) else np.nan
TIM = R["tim"]
early = TIM[(TIM.kind == "early signal") & TIM.onset_month.notna()].sort_values(["onset_month", "lift_1m"], ascending=[True, False])
early_pay = early[early.set.str.startswith("payment")]
sib = TIM[TIM.feature == "dep_sib_d6"]
sib_on = int(sib.onset_month.iloc[0]) if len(sib) and pd.notna(sib.onset_month.iloc[0]) else None
fo = R["folds"]; ahead = int((fo.auc_tree > fo.auc_simple).sum()) if "auc_tree" in fo else 0
VC = R["audit"].verdict.value_counts()
T0, T1 = R["test_months"][0], R["test_months"][-1]
NT = int(R.get("n_test", len(R["test_months"])))
NW = {6: "six", 7: "seven", 8: "eight", 9: "nine", 10: "ten", 11: "eleven", 12: "twelve"}.get(NT, str(NT))
MOST = "most" if R["bal_exit_share"] > 0.5 else "part"

def ls_val(feat, col):
    try: return float(R["ls"].loc[R["ls"].feature.str.startswith(str(R["imp"].set_index("feature").label.get(feat, "@@"))[:30]), col].iloc[0])
    except Exception: return np.nan

# 1 · KPI cards + findings
KPI = [(f"{R['exits_per_year']:,.0f}", "clients a year move their money out"),
       (usd(R["at_risk_year"]), "of normal balance leaves each year"),
       (pct(R["bal_exit_share"]), "of it leaves with the accounts still open"),
       (usd(R["main_reached"]), f"a year reachable with {K_MAIN:,} calls a month"),
       (pct(_pm.precision_mean), "of calls reach a client who is leaving"),
       (pct(R["lead3_sus_usd"]), "of those dollars flagged 3+ months ahead")]
kpi_html = "<div class='kpis'>" + "".join(f"<div class='kpi'><div class='v'>{v}</div><div class='l'>{l}</div></div>" for v, l in KPI) + "</div>"
first_sig = "; ".join(f"{plain(f, l)} {'falls' if d == 'falls' else 'rises'} from {int(-m)} months out"
                      for f, l, d, m in zip(early_pay.feature.head(3), early_pay.label.head(3), early_pay.leavers.head(3), early_pay.onset_month.head(3)))
FIND = [
 (f"About {R['exits_per_year']:,.0f} Treasury Management clients a year move their money out, holding {usd(R['at_risk_year'])} of normal balance.",
  f"{pct(R['bal_exit_share'])} of those dollars leave through <i>balance exits</i>: the client drains the balance and leaves the accounts open, so a "
  f"count of closed accounts misses {MOST} of the loss. The money is concentrated: clients with a normal balance of $10m or more account for about "
  f"{pct(big_share)} of it, and the largest single client for {pct(R['top_client_share'])}."),
 ("Payment behavior changes before the balance moves.",
  f"Compared with stayers of the same size, leavers show {first_sig or 'changes in their payments'}"
  + (f"; the balances of other entities in the same relationship start falling {abs(sib_on)} months out" if sib_on else "")
  + ". The client's own balance typically holds until the final month, when the money goes out as payments. "
  f"The model's warning builds steadily: about {pct(R['gap_ratio'])} of the final difference between leavers and stayers is already visible six months out."),
 (f"The warning list ranks leavers well and consistently.",
  f"On {NW} months the model had never seen, a call from the top {K_MAIN:,} reached a leaving client {pct(_pm.precision_mean)} of the time, "
  f"against a base rate of about {pct(R.get('base_rate', np.nan), 1)}. The advanced model beat the simple one in {ahead} of {len(fo)} months "
  f"(ranking quality {R['auc_gbm']:.3f} against {R['auc_lin']:.3f}); its dollar advantage, {usd(R['diff'])} a year, is not yet statistically proven."),
 (f"{K_MAIN:,} calls a month reach {usd(R['main_reached'])} a year of departing balance ({pct(R['main_share'])} of it).",
  f"Cautious figures: {usd(top5)} without the five largest clients, {usd(capd)} with each client's credit capped at {usd(CAP_CREDIT, 0)}. "
  f"What that saves depends on how often a call keeps the money, which only a pilot can measure: at a 5% save rate "
  f"{usd(lo * .05)}–{usd(R['main_reached'] * .05)} a year, at 20% {usd(lo * .2)}–{usd(R['main_reached'] * .2)}."),
 ("Most of the money is flagged early enough to act." if R["lead3_sus_usd"] > 0.5 else "How early the list warns.",
  f"{pct(R['lead3_sus_usd'])} of the dollars reached were on the list for at least two consecutive months, three or more months before the money left. "
  f"Most <i>clients</i> reached are small ones caught while already draining: {pct(lead1)} are first flagged in the month before the money goes."),
 ("The largest departures were checked one by one in the data.",
  f"Of {R['n_audited']} checked: {VC.get('moved to another bank', 0)} moved money to their own accounts at another bank, {VC.get('departed', 0)} departed, "
  f"{VC.get('still transacting', 0)} still transact here, {VC.get('moved within PNC', 0)} moved within PNC and {VC.get('came back', 0)} came back. "
  "The last two groups are not losses; all need an RM's confirmation.")]
find_html = "<ol class='find'>" + "".join(f"<li><b>{h}</b> {t}</li>" for h, t in FIND) + "</ol>"

# 2 · size of the problem
bands = ["< $100k", "$100k–1m", "$1m–10m", "$10m+"]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Normal balance leaving per year", "Clients leaving per year"), horizontal_spacing=.12)
for et, col in (("balance exit", C["orange"]), ("full exit", C["blue"])):
    d = AR[AR.exit_type == et].set_index(AR[AR.exit_type == et].band.astype(str)).reindex(bands)
    fig.add_bar(x=bands, y=d.per_year / 1e9, name=et.capitalize(), marker_color=col, row=1, col=1,
                hovertemplate="%{x}: $%{y:.2f}bn a year<extra>" + et + "</extra>")
    fig.add_bar(x=bands, y=d.clients_year, name=et.capitalize(), marker_color=col, row=1, col=2, showlegend=False,
                hovertemplate="%{x}: %{y:,.0f} clients a year<extra>" + et + "</extra>")
fig.update_layout(barmode="stack"); fig.update_yaxes(title_text="$bn", row=1, col=1); fig.update_yaxes(title_text="clients", row=1, col=2)
ch_size = show(fig, 430)

# 3 · what happens before a client leaves
beh = [c for c in ["amt_all_in__d6", "active_days_in__d6", "ntxn_all_in__d6", "ncpty_all_in__d6", "amt_all_out__d6", "dep_bal_vs_norm"]
       if BV is not None and c in BV.columns]
ch_beh = ""
if beh:
    fig = make_subplots(rows=2, cols=3, subplot_titles=[plain(c).replace(" (change vs 4–6 months earlier)", "") for c in beh], vertical_spacing=.18)
    for i, c in enumerate(beh):
        r_, c_ = i // 3 + 1, i % 3 + 1
        for g, col, nm in (("leaver", C["red"], "Clients who left"), ("stayer", C["blue"], "Clients who stayed")):
            d = BV[BV.grp == g].sort_values("rel")
            fig.add_scatter(x=d.rel, y=[to_display(c, v) for v in d[c]], mode="lines+markers", line=dict(color=col, width=2.5),
                            marker=dict(size=5), name=nm, showlegend=(i == 0), row=r_, col=c_,
                            hovertemplate="%{x} months before: %{y:.0f}%<extra>" + nm + "</extra>")
        fig.add_hline(y=0, line=dict(color=C["grey"], width=1), row=r_, col=c_)
    fig.update_xaxes(title_text="months before the money left"); fig.update_yaxes(title_text="% vs 4–6 months earlier", col=1)
    ch_beh = show(fig, 620)
ch_heat, ch_onset = "", ""
if LIFTM is not None and len(TIM):
    sel = list(early.feature.head(12)) + list(TIM[TIM.kind == "late warning"].feature)
    rels = sorted(LIFTM.rel.unique())
    z = [[LIFTM[(LIFTM.feature == f) & (LIFTM.rel == r)].lift.mean() for r in rels] for f in sel]
    ylab = [plain(f, l) for f, l in zip(sel, TIM.set_index("feature").label.reindex(sel))]
    fig = go.Figure(go.Heatmap(z=np.clip(np.array(z, float), 0, 4), x=rels, y=ylab, zmin=1, zmax=4,
                               colorscale=[[0, "#ffffff"], [.35, "#fde2c4"], [.7, C["orange"]], [1, C["red"]]],
                               colorbar=dict(title="× stayers"),
                               hovertemplate="%{y}<br>%{x} months before: leavers %{z:.1f}× as likely as stayers<extra></extra>"))
    fig.update_layout(title="How strongly each signal separates leavers from stayers, month by month")
    fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title_text="months before the money left")
    ch_heat = show(fig, 120 + 34 * len(sel))
    on = early.head(12).iloc[::-1]
    fig = go.Figure(go.Bar(x=-on.onset_month, y=[plain(f, l) for f, l in zip(on.feature, on.label)], orientation="h",
                           marker_color=[C["teal"] if s.startswith("payment") else C["blue"] for s in on.set],
                           hovertemplate="%{y}: fires from %{x} months before<extra></extra>"))
    fig.update_layout(title="Months of warning each signal gives (payment signals teal, deposit signals blue)")
    fig.update_xaxes(title_text="months before the money left", dtick=1)
    ch_onset = show(fig, 120 + 34 * len(on))
tg = rd("9b_total_gap"); ch_gap = ""
if tg is not None:
    fig = go.Figure(go.Scatter(x=tg.rel, y=tg.gap, mode="lines+markers", line=dict(color=C["red"], width=3), fill="tozeroy",
                               fillcolor="rgba(209,73,91,.12)", hovertemplate="%{x} months before: %{y:.2f}<extra></extra>"))
    fig.update_layout(title="How far apart the model scores leavers and stayers, month by month")
    fig.update_xaxes(title_text="months before the money left"); fig.update_yaxes(title_text="score difference (log-odds)")
    ch_gap = show(fig, 360)

# 4 · what the model looks at
fam = R["fam_imp"].copy(); fam["name"] = fam.set.map(lambda s: SETS.get(s, s.title())); fam = fam.sort_values("share").tail(14)
fig = go.Figure(go.Bar(x=fam.share * 100, y=fam.name, orientation="h", marker_color=C["blue"],
                       hovertemplate="%{y}: %{x:.1f}% of the model's weight<extra></extra>"))
fig.update_layout(title="What the model pays attention to (share of its weight, by type of behavior)"); fig.update_xaxes(title_text="%")
ch_fam = show(fig, 520)
imp = R["imp"].head(15).iloc[::-1]
fig = go.Figure(go.Bar(x=imp.share * 100, y=[plain(f, l) for f, l in zip(imp.feature, imp.label)], orientation="h", marker_color=C["teal"],
                       hovertemplate="%{y}: %{x:.1f}%<extra></extra>"))
fig.update_layout(title="The 15 individual measures the model leans on most"); fig.update_xaxes(title_text="% of the model's weight")
ch_imp = show(fig, 560)

# 5 · how well it works
fig = make_subplots(rows=1, cols=2, subplot_titles=("Hit rate: share of calls that reach a leaving client", "Share of departing dollars reached each month"))
for a, col, nm in ((0.0, C["grey"], "rank by likelihood only"), (0.5, C["blue"], "likelihood × size (recommended)"), (1.0, C["orange"], "rank by dollars at risk")):
    d = acc[(acc.model == MAIN) & (acc.alpha == a)]
    fig.add_scatter(x=d.calls, y=d.precision_mean * 100, mode="lines+markers", name=nm, line=dict(color=col, width=3), row=1, col=1,
                    hovertemplate="%{x:,} calls: %{y:.1f}%<extra>" + nm + "</extra>")
    fig.add_scatter(x=d.calls, y=d.recall_dollars * 100, mode="lines+markers", name=nm, line=dict(color=col, width=3), showlegend=False,
                    row=1, col=2, hovertemplate="%{x:,} calls: %{y:.1f}%<extra>" + nm + "</extra>")
fig.update_xaxes(title_text="calls per month", type="log", tickvals=sorted(acc.calls.unique())); fig.update_yaxes(title_text="%", rangemode="tozero")
ch_acc = show(fig, 440)
bym = R["bym"]
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Hit rate by month ({K_MAIN:,} calls)", "Ranking quality by month (AUC)"))
for m_, col, nm in (("simple", C["grey"], "Simple model"), ("tree", C["blue"], "Advanced model")):
    if f"precision_{m_}" in bym:
        fig.add_scatter(x=bym.month, y=bym[f"precision_{m_}"] * 100, mode="lines+markers", name=nm, line=dict(color=col, width=3), row=1, col=1)
        fig.add_scatter(x=bym.month, y=bym[f"auc_{m_}"], mode="lines+markers", name=nm, line=dict(color=col, width=3), showlegend=False, row=1, col=2)
fig.update_yaxes(title_text="%", rangemode="tozero", row=1, col=1); fig.update_yaxes(range=[0.5, 1], row=1, col=2)
ch_month = show(fig, 400)
cal = R["cal"]
fig = go.Figure()
fig.add_scatter(x=cal.mean_predicted * 100, y=cal.actual * 100, mode="lines+markers", name="model", line=dict(color=C["blue"], width=3),
                hovertemplate="predicted %{x:.1f}% → actually left %{y:.1f}%<extra></extra>")
mx = float(max(cal.mean_predicted.max(), cal.actual.max()) * 100)
fig.add_scatter(x=[0, mx], y=[0, mx], mode="lines", name="perfect", line=dict(color=C["grey"], dash="dash"))
fig.update_layout(title="When the model says X%, how many actually leave? (ten groups of clients)")
fig.update_xaxes(title_text="predicted chance of leaving, %"); fig.update_yaxes(title_text="actually left, %")
ch_cal = show(fig, 400)
cmp = R["cmp"]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Dollars reached per year ($bn)", "Hit rate (%)"))
for m_, col, nm in (("simple", C["grey"], "Simple model"), ("tree", C["blue"], "Advanced model")):
    if m_ in cmp:
        fig.add_bar(x=cmp.calls.astype(str), y=cmp[m_] / 1e9, name=nm, marker_color=col, row=1, col=1)
        fig.add_bar(x=cmp.calls.astype(str), y=cmp[f"precision_{m_}"] * 100, name=nm, marker_color=col, showlegend=False, row=1, col=2)
fig.update_xaxes(title_text="calls per month")
ch_cmp = show(fig, 400)

# 6 · how early
ls_ = R["lead_sus"].copy()
fig = go.Figure()
fig.add_bar(x=ls_.lead_band.astype(str), y=ls_.share_clients * 100, name="share of clients", marker_color=C["grey"])
fig.add_bar(x=ls_.lead_band.astype(str), y=ls_.share_dollars * 100, name="share of dollars", marker_color=C["blue"])
fig.update_layout(title="Months between a sustained alert and the money leaving", barmode="group")
fig.update_xaxes(title_text="months of warning"); fig.update_yaxes(title_text="%")
ch_lead = show(fig, 400)

# 7 · value
v_ = val[val.model == MAIN]
fig = go.Figure()
for a, col, nm in ((0.0, C["grey"], "likelihood only"), (0.5, C["blue"], "likelihood × size (recommended)"), (1.0, C["orange"], "dollars at risk")):
    d = v_[v_.alpha == a]
    fig.add_scatter(x=d.calls, y=d.dollars_reached_year / 1e9, mode="lines+markers", name=nm, line=dict(color=col, width=3),
                    hovertemplate="%{x:,} calls: $%{y:.1f}bn a year<extra>" + nm + "</extra>")
fig.update_layout(title="Departing balance reached per year, by number of calls per month")
fig.update_xaxes(title_text="calls per month", type="log", tickvals=sorted(v_.calls.unique())); fig.update_yaxes(title_text="$bn a year", rangemode="tozero")
ch_val = show(fig, 420)
scen = [("All departing clients", R["main_reached"]), ("Without the 5 largest", top5), (f"Credit capped at {usd(CAP_CREDIT, 0)}", capd)]
fig = go.Figure()
for (nm, base), col in zip(scen, [C["blue"], C["sky"], C["teal"]]):
    fig.add_bar(x=[f"{int(s * 100)}%" for s in SAVE_RATES], y=[base * s / 1e6 for s in SAVE_RATES], name=nm, marker_color=col,
                hovertemplate="save rate %{x}: $%{y:,.0f}m a year<extra>" + nm + "</extra>")
fig.update_layout(title=f"Potential savings per year at {K_MAIN:,} calls a month, by save rate", barmode="group")
fig.update_xaxes(title_text="share of reached clients whose money we keep (save rate)"); fig.update_yaxes(title_text="$m a year")
ch_save = show(fig, 420)

# 8 · largest departures
acn = R["audit_counts"]
VCOL = {"moved to another bank": C["red"], "departed": C["orange"], "still transacting": C["gold"], "moved within PNC": C["teal"], "came back": C["sky"]}
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "domain"}, {"type": "domain"}]], subplot_titles=("Clients", "Normal balance"))
fig.add_pie(labels=acn.verdict, values=acn.clients, hole=.55, marker_colors=[VCOL.get(v, C["grey"]) for v in acn.verdict], row=1, col=1)
fig.add_pie(labels=acn.verdict, values=acn.normal, hole=.55, marker_colors=[VCOL.get(v, C["grey"]) for v in acn.verdict], row=1, col=2,
            hovertemplate="%{label}: $%{value:,.0f}<extra></extra>")
ch_aud = show(fig, 380)
aud = R["audit"].copy()
aud_tbl = table(aud.sort_values("normal", ascending=False), ["client", "exit_type", "exit_month", "normal", "verdict"],
                ["client ref", "how the money left", "month", "normal balance", "what the data shows"], {"normal": lambda v: usd(v)})

# 9 · client stories
KIND = {"Early catch": "The list flagged this client well before the money moved.",
        "Payments moved before the balance": "Payment behavior changed while the balance still looked normal.",
        "Drained without closing": "The client emptied its balance but kept the accounts open.",
        "Late catch": "Flagged only shortly before the money left: the kind of case where time to act is short.",
        "New money that left": "A large balance arrived recently and left again within the year.",
        "False alarm (most confident wrong call)": "The client the model was most sure about that did not fully leave.",
        "Missed": "The largest departure the list never flagged."}
stories = []
if LONG is not None:
    for n in sorted(LONG.example.unique()):
        if n in EXAMPLES_SKIP: continue
        L = LONG[LONG.example == n]; kind = L.kind.iloc[0]; cref = L.client.iloc[0]
        months = list(dict.fromkeys(L.sort_values("month").month)); ix = {m: i for i, m in enumerate(months)}
        g = lambda var: L[L.variable == var].set_index("month").value.reindex(months)
        bal = g("balance"); norm = float(L[L.variable == "normal"].value.iloc[0])
        onl = L[L.variable == "score"].set_index("month").fired.reindex(months).map(lambda x: bool(x) if pd.notna(x) and str(x) != "None" else False)
        vx = L[L.variable == "balance"].set_index("month").vs_exit.reindex(months)
        exit_m = next((m for m in months if vx.get(m) == 0), None)
        on_m = [m for m in months if onl[m]]
        sus_m = next((m for m, m2 in zip(months, months[1:]) if onl[m] and onl[m2]), on_m[0] if on_m else None)
        trig = [v for v in dict.fromkeys(L.variable) if v not in ("balance", "normal", "score")]
        rows_ = 1 + len(trig)
        fig = make_subplots(rows=rows_, cols=1, shared_xaxes=True, vertical_spacing=.06,
                            row_heights=[.4] + [.6 / max(len(trig), 1)] * len(trig),
                            subplot_titles=["Balance ($m) against normal"] + [f"{plain(v, L[L.variable == v].label.iloc[0])} — {unit(v)}" for v in trig])
        xs = list(range(len(months)))
        fig.add_scatter(x=xs, y=bal / 1e6, mode="lines+markers", line=dict(color=C["blue"], width=3), name="balance",
                        customdata=months, hovertemplate="%{customdata}: $%{y:,.1f}m<extra></extra>", row=1, col=1)
        fig.add_hline(y=norm / 1e6, line=dict(color=C["grey"], dash="dash"), row=1, col=1)
        for m in on_m:
            fig.add_vrect(x0=ix[m] - .5, x1=ix[m] + .5, fillcolor=C["orange"], opacity=.15, line_width=0)
        if exit_m: fig.add_vline(x=ix[exit_m], line=dict(color=C["red"], dash="dash", width=2))
        fired_first = []
        for r_, v in enumerate(trig, start=2):
            Lv = L[L.variable == v].set_index("month").reindex(months)
            y = [to_display(v, x) for x in Lv.value]; th = to_display(v, Lv.threshold.dropna().iloc[0]) if Lv.threshold.notna().any() else None
            fig.add_scatter(x=xs, y=y, mode="lines+markers", line=dict(color=C["ink"], width=2), marker=dict(size=4), showlegend=False,
                            customdata=months, hovertemplate="%{customdata}: %{y:.1f}<extra></extra>", row=r_, col=1)
            if th is not None:
                fig.add_hline(y=th, line=dict(color=C["red"], dash="dot", width=1), row=r_, col=1)
            fm = [m for m in months if bool(Lv.fired.get(m))]
            fig.add_scatter(x=[ix[m] for m in fm], y=[y[ix[m]] for m in fm], mode="markers", marker=dict(color=C["red"], size=9),
                            showlegend=False, hoverinfo="skip", row=r_, col=1)
            pre = [m for m in fm if exit_m is None or ix[m] < ix[exit_m]]
            if pre: fired_first.append((ix[pre[0]], v))
        fig.update_xaxes(tickvals=xs, ticktext=months, tickangle=-45, row=rows_, col=1)
        fig.update_layout(showlegend=False, title=f"Client {cref} · {kind}")
        chart = show(fig, 330 + 170 * len(trig))
        fired_first.sort()
        sus_i = ix.get(sus_m) if sus_m else None
        bits = [KIND.get(kind, "")]
        if kind.startswith("False"):
            after = bal.iloc[sus_i:] if sus_i is not None else bal
            low = float(after.min() / norm) if len(after) and norm else np.nan
            bits.append(f"On the list from {sus_m} and the balance never fully left." +
                        (f" But it fell to {pct(low)} of normal: this client moved a large part of its money without fully leaving — a <b>partial leaver</b>, "
                         "which the definition of attrition used here does not count." if pd.notna(low) and low < 0.7 else ""))
        elif kind == "Missed":
            bits.append(f"The money left in {exit_m}; the client never reached the top {K_MAIN:,}.")
        elif sus_m:
            lead = ix[exit_m] - sus_i if exit_m else None
            bits.append(f"On the list from <b>{sus_m}</b>" + (f", <b>{lead} month{'s' if lead != 1 else ''}</b> before the money left in {exit_m}" if lead else "") +
                        f". Balance then {usd(bal.iloc[sus_i])} against a normal {usd(norm)} ({pct(bal.iloc[sus_i] / norm)}).")
        if fired_first:
            bits.append("Signals, in the order they fired: " + "; ".join(
                f"{plain(v, L[L.variable == v].label.iloc[0])} ({months[i]}" + (f", {ix[exit_m] - i} months before the exit)" if exit_m else ")")
                for i, v in fired_first[:4]) + ".")
            if fired_first[0][1].startswith("dep_sib"):
                bits.append("Other entities in the same relationship moved their money first — this client followed.")
        if not kind.startswith("False") and kind != "Missed" and (fired_first or trig):
            bits.append(f"<span class='play'>Suggested conversation: {play((fired_first[0][1] if fired_first else trig[0]))}</span>")
        stories.append(f"<div class='story'><div class='story-h'><span class='tag'>{esc(kind)}</span> Client {esc(cref)}</div>"
                       f"<p>{' '.join(b for b in bits if b)}</p>{chart}</div>")

# ── assemble ─────────────────────────────────────────────────────────
SECTIONS = [
 ("summary", "Summary", f"""{kpi_html}<div class='lead'>This report explains how Treasury Management deposit clients leave, what their payments
 show in the months before, and how a monthly warning list can reach them in time. All figures come from {NW} months ({T0} to {T1}) that the
 model had never seen when it made its predictions.</div>{find_html}"""),
 ("definitions", "What we mean by attrition", f"""
 <p>We say a client's <b>money leaves</b> when either of two things happens, whichever comes first:</p>
 <div class='cards2'><div class='card'><h4>Full exit</h4><p>Every deposit account is closed or inactive, and stays that way for at least two
 months. Fewer than {pct(R.get('p_return', 0), 1)} of such clients ever come back.</p></div>
 <div class='card'><h4>Balance exit</h4><p>The balance falls below 5% of the client's normal level for three months in a row, while the accounts
 stay open. The relationship looks alive on paper, but the money is gone.</p></div></div>
 <p>A client's <b>normal balance</b> is the middle value of its balance over the previous twelve months, so a single large deposit or withdrawal
 does not distort it. We only put clients on the list while their <b>money is still here</b> — at least half of their normal balance, and a normal
 balance of at least $10,000 — because a call can only save money that has not yet left.</p>
 <p>The model predicts whether the money will leave in the <b>next six months</b>: long enough for an RM to act, short enough that the warning is
 about a decision the client is making now.</p>
 <div class='callout'>Why count balance exits? {pct(R['bal_exit_share'])} of the dollars that leave do so with the accounts still open. A measure based only on
 closed accounts would miss {MOST} of the loss.</div>"""),
 ("size", "How big the problem is", f"""
 <p>About <b>{R['exits_per_year']:,.0f} clients</b> a year move their money out, taking <b>{usd(R['at_risk_year'])}</b> of normal balance. Most leavers are small,
 but most of the money sits with a few hundred large clients: those with $10m or more of normal balance carry about {pct(big_share)} of the dollars.</p>
 {ch_size}<div class='callout'>For RMs this means two different jobs: a few large relationships where one conversation can protect hundreds of
 millions, and many small clients where a lighter, earlier touch is needed.</div>"""),
 ("before", "What happens before a client leaves", f"""
 <p>We lined up every client who left on the month their money went, and compared them with clients of the same size who stayed. The charts show
 the typical change against the same client's own level four to six months earlier.</p>{ch_beh}
 <p>The pattern is consistent: <b>money coming in dries up first</b> — fewer days with receipts, fewer incoming payments, fewer payers — while the
 balance still looks normal. In the final month the balance drops and the money goes out as payments.</p>
 <h3>Which signals fire, and when</h3><p>A signal <i>fires</i> when a client's value moves further than it does for nine in ten stayers of the same
 size. The darker the cell, the more often leavers show the signal compared with stayers.</p>{ch_heat}{ch_onset}
 <p>The model's overall view builds the same way: the gap between leavers' and stayers' scores widens steadily over the last year.</p>{ch_gap}"""),
 ("model", "How the warning list works", f"""
 <p>Every month each client with money still here gets a <b>score</b>: the chance its money leaves within six months. The score comes from about
 {R.get('n_feats', 300):,} measures of deposit and payment behavior — levels, changes over three and six months, and comparisons with clients of a
 similar size. Clients are ranked by <b>likelihood × size</b> (the square root of the normal balance), so a likely leaver with a large balance comes
 before an equally likely one with a small balance, without letting size alone decide. The top {K_MAIN:,} make the list.</p>
 <div class='cards3'><div class='card'><h4>1 · Score</h4><p>Chance the money leaves in the next six months, from the client's own history.</p></div>
 <div class='card'><h4>2 · Rank</h4><p>Likelihood × size, so the calls go where the most money can be kept.</p></div>
 <div class='card'><h4>3 · Explain</h4><p>For every client, the three measures that raised its score most — the reason codes an RM sees.</p></div></div>
 {ch_fam}{ch_imp}"""),
 ("accuracy", "How well it works", f"""
 <p>Every figure here is <b>out of sample</b>: for each of the {NW} test months the model was built only from what was known before that month,
 exactly as it would run in practice.</p>{ch_acc}
 <p>At {K_MAIN:,} calls a month, <b>{pct(_pm.precision_mean)}</b> of calls reach a client who is about to leave — roughly one in {max(int(round(1 / _pm.precision_mean)), 1)},
 against about one in {max(int(round(1 / R.get('base_rate', .05))), 1)} for a random client. Ranking purely by likelihood gives more hits but reaches
 far fewer dollars; the recommended ranking reaches <b>{pct(float(acc[(acc.model == MAIN) & (acc.alpha == A_MAIN) & (acc.calls == K_MAIN)].recall_dollars.iloc[0]))}</b>
 of the departing dollars each month.</p>{ch_month}
 <p>The results are stable month to month: the best month's hit rate is {R['prec_ratio']:.1f}× the worst.</p>{ch_cal}
 <p>The probabilities are usable as stated: when the model says a client has a one-in-five chance, about that share leave.</p>
 <h3>Simple or advanced model?</h3>{ch_cmp}<p>The advanced (gradient-boosted tree) model beats the simple (regression) model in {ahead} of {len(fo)}
 months, and reaches {usd(R['diff'])} more a year at {K_MAIN:,} calls. That difference is not yet statistically certain
 (95% range {usd(R['diff_ci'][0])} to {usd(R['diff_ci'][1])}); both are good enough to run a pilot.</p>"""),
 ("early", "How early the list warns", f"""
 <p>A client on the list for two consecutive months has a <b>sustained alert</b>. {pct(R['lead3_sus_usd'])} of the dollars reached had a sustained alert
 three or more months before the money left.</p>{ch_lead}
 <div class='callout'>Many small clients appear only once, in the month before their money goes ({pct(lead1)} of clients reached): by the next month
 the money has already left, so they cannot appear twice. For those, the list is a last-chance call; for large relationships it is an early warning.</div>"""),
 ("value", "What it could be worth", f"""{ch_val}
 <p>At {K_MAIN:,} calls a month the list reaches <b>{usd(R['main_reached'])}</b> a year of departing balance. The figure leans on a few very large
 clients, so we also show it without the five largest ({usd(top5)}) and with each client's credit capped at {usd(CAP_CREDIT, 0)} ({usd(capd)}).</p>
 {ch_save}<div class='callout'>Reaching a client is not the same as keeping the money. The <b>save rate</b> — how often a call keeps the balance —
 cannot be measured from history. A pilot with a comparison group is the only way to know it; the chart shows what 1% to 20% would be worth.</div>"""),
 ("largest", "The largest departures, checked one by one", f"""
 <p>Because a few clients carry so much of the money, each of the largest departures was checked in the data: did the money come back, move to
 another PNC entity in the same relationship, go to the client's own accounts at another bank, or does the client still transact here?</p>
 {ch_aud}{aud_tbl}<p class='note'>Client references are anonymous codes; the matching list is held by the analytics team for RM follow-up.</p>"""),
 ("stories", "Client stories", """
 <p>These clients were chosen by fixed rules, not hand-picked. Each chart shows the balance against its normal level (dashed), the months the client
 was on the list (shaded), the month the money left (red line), and the signals that fired (red dots beyond the dotted threshold).</p>""" + "".join(stories)),
 ("playbook", "What an RM can do", """
 <table class='t'><thead><tr><th>When the signal is…</th><th>It usually means…</th><th>A good first conversation</th></tr></thead><tbody>
 <tr><td>Fewer receipts, fewer payers, fewer days with money coming in</td><td>The client's customers are paying it less, or paying into another bank</td>
 <td>Ask about collections and receivables; review lockbox and remote deposit set-up</td></tr>
 <tr><td>New banks in its payments; money sent to its own accounts elsewhere; shift to wires</td><td>A competitor is being onboarded</td>
 <td>Compare pricing, earnings credit and services; prepare a retention offer</td></tr>
 <tr><td>Other entities in the same relationship moving money</td><td>A group-level decision is under way</td>
 <td>Speak to the relationship's decision maker, not only this entity</td></tr>
 <tr><td>Fewer payments overall and fewer trading partners</td><td>The business itself is shrinking</td>
 <td>Check on the business; offer credit, working capital or payment products</td></tr>
 <tr><td>Balance falling within the month, month-end below average</td><td>The money is leaving now</td>
 <td>Call immediately; ask about cash plans and offer sweep or liquidity options</td></tr></tbody></table>"""),
 ("limits", "What to keep in mind", """<ul class='limits'>
 <li><b>The save rate is unknown.</b> Savings figures are illustrations until a pilot measures how often a call keeps the money.</li>
 <li><b>Partial leavers are not counted.</b> A client that moves half its money is not an "exit" here, so some apparent false alarms are real losses.</li>
 <li><b>A few clients dominate the dollars.</b> Look at the cautious figures alongside the headline.</li>
 <li><b>History is short.</b> The data start in January 2024; every prediction needs six months of outcome to be checked, which leaves """ + NW + """ test months.</li>
 <li><b>Client stories and the largest departures need RM review</b> before they are quoted outside this group.</li></ul>"""),
 ("next", "Next steps", """<ol class='find'>
 <li><b>Pilot.</b> Give RMs the monthly list for a subset of clients and keep a comparable group without calls, to measure the save rate.</li>
 <li><b>RM feedback.</b> Review the client stories and the largest departures with the RMs who know them.</li>
 <li><b>Partial leavers.</b> Add a measure for clients that move a large part of their money without fully leaving.</li>
 <li><b>Monthly refresh.</b> Rebuild the list each month as new data arrive.</li></ol>"""),
 ("glossary", "Glossary", table(pd.DataFrame([
   ("Money leaves", "A full exit or a balance exit, whichever comes first."),
   ("Full exit", "Every deposit account closed or inactive for at least two months."),
   ("Balance exit", "Balance below 5% of normal for three months while the accounts stay open."),
   ("Normal balance", "The middle value of the client's balance over the previous 12 months."),
   ("Money still here", "Balance at least half of normal, and normal balance at least $10,000. Only these clients are listed."),
   ("Score", "The model's estimate of the chance the money leaves within six months."),
   ("Likelihood × size", "Ranking by score times the square root of the normal balance, so large and likely leavers come first."),
   ("Hit rate", "Share of listed clients whose money actually left within six months."),
   ("Dollars reached", "Balance of leaving clients at the time they were first listed (never more than their normal balance), per year."),
   ("Save rate", "Share of reached clients whose money a call keeps. Unknown until a pilot."),
   ("Sustained alert", "On the list for at least two consecutive months."),
   ("Signal fires", "A client's value moves further than it does for nine in ten stayers of the same size."),
   ("Similar-sized clients", "Clients in the same tenth of normal balance in the same month."),
   ("Ranking quality (AUC)", "Chance the model scores a random leaver above a random stayer: 0.5 is guessing, 1.0 is perfect."),
   ("Out of sample", "Tested on months the model had never seen, using only information available at the time.")],
   columns=["term", "meaning"]), ["term", "meaning"], ["Term", "Meaning"], {})),
 ("method", "How the analysis was done", f"""<p class='note'>For readers who want the detail.</p><ul class='limits'>
 <li><b>Data.</b> Daily deposit balances and account status for Treasury Management clients, and each client's monthly payment profile from the
 Payment Knowledge Graph (who it pays and receives from, how, through which banks), from January 2024 to {R.get('dep_end', '')}.</li>
 <li><b>Measures.</b> About {R.get('n_feats', 300):,} per client and month: balance level and trend, account activity, pricing, relationship,
 and for each payment measure its change over three and six months and its difference from similar-sized clients. Near-duplicate measures were removed.</li>
 <li><b>Models.</b> A regression model and a gradient-boosted tree model, compared on identical data. The tree model's probabilities are calibrated
 on recent held-out clients.</li>
 <li><b>Testing.</b> Rolling monthly: each test month is predicted by a model trained only on outcomes known before it. A fifth of clients are held out
 during training for tuning. Uncertainty ranges come from resampling the departing clients.</li></ul>"""),
]
nav = "".join(f"<a href='#{i}'>{t}</a>" for i, t, _ in SECTIONS)
body = "".join(f"<section id='{i}'><h2>{t}</h2>{c}</section>" for i, t, c in SECTIONS)
CSS = f"""
:root{{--navy:{C['navy']};--blue:{C['blue']};--teal:{C['teal']};--orange:{C['orange']};--ink:{C['ink']};--light:{C['light']}}}
*{{box-sizing:border-box}}body{{margin:0;font-family:'Segoe UI',Helvetica,Arial,sans-serif;color:var(--ink);background:#fff;line-height:1.55}}
header{{background:linear-gradient(120deg,var(--navy),var(--blue));color:#fff;padding:48px 64px 40px}}
header h1{{margin:0;font-size:34px;font-weight:600}}header p{{margin:8px 0 0;opacity:.85;font-size:17px}}
header .meta{{margin-top:18px;font-size:13px;opacity:.7}}
.wrap{{display:flex}}nav{{position:sticky;top:0;align-self:flex-start;width:230px;min-width:230px;height:100vh;overflow:auto;
padding:28px 18px;background:var(--light);border-right:1px solid #e3e8ef}}
nav a{{display:block;padding:7px 10px;margin:2px 0;color:var(--navy);text-decoration:none;border-radius:6px;font-size:14px}}
nav a:hover{{background:#dde6f2}}main{{flex:1;max-width:1180px;padding:24px 56px 80px}}
section{{padding-top:18px;border-bottom:1px solid #eef1f5;padding-bottom:26px}}h2{{color:var(--navy);font-size:26px;margin:18px 0 12px}}
h3{{color:var(--blue);margin-top:26px}}h4{{margin:0 0 6px;color:var(--navy)}}
.kpis{{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;margin:18px 0 22px}}
.kpi{{background:#fff;border:1px solid #e3e8ef;border-top:4px solid var(--teal);border-radius:10px;padding:16px 18px;box-shadow:0 2px 6px rgba(11,37,69,.06)}}
.kpi .v{{font-size:30px;font-weight:700;color:var(--navy)}}.kpi .l{{font-size:14px;color:#52606d}}
.lead{{font-size:17px;color:#334e68;margin:8px 0 14px}}ol.find li{{margin:12px 0}}
.callout{{background:#fff7ec;border-left:5px solid var(--orange);padding:14px 18px;border-radius:6px;margin:16px 0}}
.cards2,.cards3{{display:grid;gap:16px;margin:16px 0}}.cards2{{grid-template-columns:1fr 1fr}}.cards3{{grid-template-columns:1fr 1fr 1fr}}
.card{{background:var(--light);border-radius:10px;padding:16px 18px}}.card p{{margin:0}}
.chart{{margin:14px 0;border:1px solid #eef1f5;border-radius:10px;padding:6px}}
table.t{{border-collapse:collapse;width:100%;font-size:14px;margin:12px 0}}table.t th{{background:var(--navy);color:#fff;text-align:left;padding:8px 10px}}
table.t td{{padding:7px 10px;border-bottom:1px solid #e9edf2;vertical-align:top}}table.t tr:nth-child(even) td{{background:#fafbfd}}
.story{{border:1px solid #e3e8ef;border-radius:12px;padding:18px 22px;margin:22px 0;box-shadow:0 2px 8px rgba(11,37,69,.05)}}
.story-h{{font-size:18px;font-weight:600;color:var(--navy);margin-bottom:6px}}
.tag{{display:inline-block;background:var(--blue);color:#fff;border-radius:14px;padding:2px 12px;font-size:13px;margin-right:8px}}
.play{{display:block;margin-top:10px;background:#e8f6f4;border-left:4px solid var(--teal);padding:10px 14px;border-radius:6px}}
.note{{color:#6b7785;font-size:13px}}ul.limits li{{margin:8px 0}}
footer{{padding:24px 64px;color:#8a96a3;font-size:12px;border-top:1px solid #eef1f5}}
@media(max-width:900px){{nav{{display:none}}main{{padding:16px}}.kpis,.cards2,.cards3{{grid-template-columns:1fr}}}}"""
js = f"<script>{get_plotlyjs()}</script>" if EMBED_PLOTLY else "<script src='https://cdn.plot.ly/plotly-2.35.2.min.js'></script>"
doc = f"""<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>
<title>{esc(TITLE)}</title><style>{CSS}</style>{js}</head><body>
<header><h1>{esc(TITLE)}</h1><p>{esc(SUBTITLE)}</p><div class='meta'>Treasury Management · Data Science · Payment Knowledge Graph ·
test months {T0} to {T1} · prepared {datetime.date.today():%B %d, %Y} · internal</div></header>
<div class='wrap'><nav>{nav}</nav><main>{body}</main></div>
<footer>Internal — PNC Treasury Management. Figures are estimates from historical data; client references are anonymous.</footer></body></html>"""
out = RUN / OUT_FILE
out.write_text(doc, encoding="utf-8")
print(f"written {out.resolve()}  ({out.stat().st_size / 1e6:,.1f} MB, {len(stories)} client stories, Plotly {plotly.__version__})")